### Exercises

#### Exercise 1

Read in the simulated gene expression dataset found in `gene_data.csv`. Then fit a multiple linear regression model and compute the RMSE for this model.

In [19]:
import pandas as pd
df = pd.read_csv("../data/gene_data.csv")
df.head()

,TP53,BRCA1,BRCA2,EGFR,MYC,KRAS,PTEN,APOE,TNF,VEGFA,...,PECAM1,VCAM1,IFNG,IL10,TGFB1,NOS3,PPARG,SREBF1,FASN,Trait
0,8.545177,1.930207,4.378380,4.026826,24.580835,4.443665,10.133618,44.363948,10.012965,13.597512,...,10.253655,8.300915,4.494035,5.102598,8.070732,2.742337,8.269346,9.127478,12.664307,397.425898
1,7.268998,7.739082,4.260046,187.088252,2.428403,5.527279,4.123282,6.489095,6.208605,4.678597,...,1.636358,3.702188,0.490370,3.770221,5.056198,2.061478,6.180184,44.558453,6.914127,423.246711
2,7.209494,10.807883,3.718422,7.500737,35.877357,5.237345,15.770932,9.650083,11.374552,7.330080,...,12.560870,3.541525,10.302735,18.979333,8.390883,1.193579,10.834315,4.480203,9.926243,366.093684
3,16.205869,12.612047,3.420913,66.814748,10.542371,5.922275,20.534795,217.900224,3.087735,1.801208,...,0.895400,15.023451,5.971124,6.261811,10.234016,72.412861,7.233361,18.423569,2.219636,407.067138
4,6.491159,4.214773,5.366326,1.062904,77.480568,6.173582,9.439733,13.281299,21.362738,3.986367,...,2.490854,15.439841,5.188251,7.293647,12.292569,13.123096,8.388838,4.485446,10.787298,343.912618


In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
y = df["Trait"]
X = df.drop(columns="Trait")
full_model = LinearRegression()
full_model.fit(X, y)
full_preds = full_model.predict(X)
print("RMSE: ", root_mean_squared_error(y, full_preds))

RMSE:  5.7604843529814405


#### Exercise 2

##### Part 1

Split the data into training and test datasets using 20% of the data for testing and 80% for training.

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

##### Part 2

Fit the multiple linear regression model to the training data and then compute the test RMSE for the test dataset.

In [22]:
train_model = LinearRegression()
train_model.fit(X_train, y_train)
test_preds = train_model.predict(X_test)
print("RMSE: ", root_mean_squared_error(y_test, test_preds))

RMSE:  236.06370048643942


##### Part 3

What do you observed about the train RMSE vs the test RMSE? Why do you think this is happening?

In [23]:
from sklearn.linear_model import Lasso
lasso_model = Lasso(alpha=1)
lasso_model.fit(X_train, y_train)
lasso_preds = lasso_model.predict(X_test)
print("RMSE: ", root_mean_squared_error(y_test, lasso_preds))

RMSE:  30.611570988475837


#### Exercise 3

##### Part 1

Use the `StandardScaler` class to compute means and standard deviations for each predictor in the training dataset.

In [25]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)

StandardScaler()

##### Part 2

Standardize the predictors in the test dataset using the means and standard deviations computed in Part 1 with the `StandardScaler` class.

In [26]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
scaled_model = Lasso(alpha=1)
scaled_model.fit(X_train_scaled, y_train)
scaled_preds = scaled_model.predict(X_test_scaled)
print("RMSE: ", root_mean_squared_error(y_test, scaled_preds))

RMSE:  20.690437414750967


#### Exercise 4

##### Part 1

Utilize the `LassoCV` class to use cross-validation to find the best value for $\alpha$ in the LASSO regression model and then compute the RMSE for the selected model.

In [27]:
from sklearn.linear_model import LassoCV
lasso_CV_model = LassoCV(cv=10)
lasso_CV_model.fit(X_train_scaled, y_train)
lasso_CV_preds = lasso_CV_model.predict(X_test_scaled)
print("Selected alpha: ", lasso_CV_model.alpha_)
print("RMSE: ", root_mean_squared_error(y_test, lasso_CV_preds))

Selected alpha:  0.5771480163361549
RMSE:  22.378538322492794


##### Part 2

In your own words, explain what `LassoCV` is doing. Then explain the difference between train, validation, and test datasets.

LassoCV cuts data sets into a specified amount and then computes the average validation MSE and the cross-validation error. The training data is the given data used to make the model. The validation data is used then after to make the model better and more accurate. The testing data is run at the end to make sure it is working correctly and accurately.

##### Part 3

What is overfitting? How does LASSO regression utilize regularization to prevent overfitting?

When the alpha is too close to 0, and LASSO uses alphas that are not too close to 0

##### Part 4

What exactly is the LASSO objective function designed to do and how does it prevent overfitting?

The LASSO objective function prevents alpha values close to 0 from being chosen and this also prevents overfitting.

##### Part 5

Extract the coefficient estimates from the fitted model to determine which of the predictors are "active" (non-zero).

In [28]:
{X.columns[i] : lasso_CV_model.coef_[i] for i in range(len(X.columns))}

{'TP53': -0.0,
 'BRCA1': -0.0,
 'BRCA2': 0.0,
 'EGFR': 0.0,
 'MYC': -0.3509641085479791,
 'KRAS': 0.0,
 'PTEN': 0.0,
 'APOE': 1.0902661893023848,
 'TNF': -0.0,
 'VEGFA': 0.5208539367098591,
 'IL6': 0.0,
 'MTHFR': 5.159913500873061,
 'AKT1': 0.0,
 'ESR1': 0.0,
 'CDKN2A': 0.0,
 'MAPK1': -0.0,
 'JAK2': -0.0,
 'STAT3': 0.0,
 'MTOR': 0.2722197706564005,
 'CTNNB1': -0.0,
 'PIK3CA': -0.0,
 'BRAF': 0.005134060066280811,
 'ERBB2': -3.3812958521507976,
 'SMAD4': 3.0642509980236494,
 'FGFR1': -0.18533384167880224,
 'KIT': 0.8169307215754984,
 'PDGFRA': 0.26890928794510366,
 'RB1': -7.5393570838413275,
 'NOTCH1': 1.3956991277904498,
 'GATA3': -0.0,
 'FOXP3': -0.32052977821188644,
 'NFKB1': 0.0,
 'HIF1A': 0.9232236578394571,
 'CXCL8': -0.0,
 'CCND1': -2.1198457092704457,
 'ABL1': -0.45415896924853555,
 'CDH1': -41.36258374723637,
 'VHL': 5.816970554603645,
 'FLT3': 0.3138909119497068,
 'ATM': 0.5415382728828751,
 'PALB2': -2.194596907904642,
 'AR': -1.4209721601143075,
 'SMARCA4': 0.0,
 'IDH1': 1.7